### Importing libraries

In [68]:
import os 
from openai import OpenAI 
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display

### Connecting to OpenAI or Ollama 

In [70]:
load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')
ed_site = "https://edwarddonner.com"

if not api_key:
    print('No API key found')
elif not api_key.startswith('sk-proj'):
    print('API key found, but does not start with sk-proj')
elif api_key != api_key.strip():
    print('API key found, looks like it might have space or tab characters')
else:
    print('API key found and good so far')


API key found, but does not start with sk-proj


## Making a call to a Frontier model

In [71]:
base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

message = 'hello, this is a test message to you'

messages = [{'role': 'user', 'content': message}]
messages

[{'role': 'user', 'content': 'hello, this is a test message to you'}]

In [72]:
openai = OpenAI(api_key=api_key, base_url=base_url)
response = openai.chat.completions.create(model='gemini-2.5-flash-lite', messages = messages)
response.choices[0].message.content

"Hello! Thank you for the test message. I received it loud and clear.\n\nHow can I help you today? Is there anything specific you'd like to test or ask me?"

### 1st project

In [73]:
ed_response = fetch_website_contents(ed_site)
print(ed_response)

Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 500,000 enrolled across 194

## Types of prompts

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [74]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""

### Messages 

In [ ]:
messages = [
    {"role": "system", "content": "You are an helpful assistant"},
    {"role": "user", "content": "What is 2 + 8?"}
]

response = openai.chat.completions.create(model='gemini-2.5-flash-lite', messages=messages)
response.choices[0].message.content

"Oh, wow, a real brain-buster. Let me put on my thinking cap for this one.\n\nAfter extensive deliberation and a rigorous mental workout, I've concluded that 2 + 8 equals... drumroll please... **10**.\n\nTry not to hurt yourself with that kind of intense calculation."

## Summarize the website data
### using system-prompt, and user-prompt

In [76]:
def messages_for(website):
    return [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt_prefix + website}
    ]

In [78]:
messages_for(ed_response)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\nHome - Edward Donner\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. W

## Time to call the API using the data

In [79]:
# call the API now.

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model='gemini-2.5-flash-lite',
        messages=messages_for(website)
    )
    return response.choices[0].message.content


In [80]:
summarize('https://edwarddonner.com')

'So, Edward Donner is an AI enthusiast who apparently *loves* talking about LLMs so much that his friends made him create online courses. Shockingly, they became super popular. He\'s also the CTO of Nebula.io, which sounds like another AI venture, and previously founded untapt, which was acquired. Oh, and he\'s built a game called "Outsmart" where AI models battle it out, because apparently, digital diplomacy is a thing now. He\'s also "very amateur" at music production, which is probably for the best.'

### Trying with more websites 
#### Websites rendered using JS, react js won't show up. selenium will be needed, check community-contributions

In [81]:
summarize("https://anthropic.com")

'So, Anthropic is apparently trying to make AI safe, which is… novel. They\'ve got research, products like Claude (which sounds suspiciously like a chatbot that won\'t show you ads, shocking!), and even something called "Claude\'s Constitution." Oh, and they just dropped Claude Opus 4.7, which is apparently better at coding, agents, vision, and "complex professional work" – so basically, it can do all your chores and then some. They also announced that Claude is a "space to think," which, if it means what I think it means, is just a fancy way of saying it\'s a chatbot without annoying ads. Groundbreaking.'

In [82]:
summarize("https://openai.com/about/")

"So, OpenAI is here to make sure AI doesn't, you know, *end* us all. They're building super-smart AI, which they call AGI, and apparently, they've got a whole charter and everything to prove they're *not* secretly plotting world domination. They've also got some fancy new AI for life sciences called GPT-Rosalind, and they're keeping a close eye on their coding AI to make sure it doesn't go rogue. Oh, and they've got a nonprofit and a for-profit arm, because *of course* they do. If you're looking to join the AI revolution (or just curious about their latest GPT-something-or-other), they're hiring."

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise, you experienced calling the Cloud API of a Frontier Model (a leading model at the frontier of AI) for the first time. We will be using APIs like OpenAI at many stages in the course, in addition to building our own LLMs.

More specifically, we've applied this to Summarization - a classic Gen AI use case to make a summary. This can be applied to any business vertical - summarizing the news, summarizing financial performance, summarizing a resume in a cover letter - the applications are limitless. Consider how you could apply Summarization in your business, and try prototyping a solution.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you continue - now try yourself</h2>
            <span style="color:#900;">Use the cell below to make your own simple commercial example. Stick with the summarization use case for now. Here's an idea: write something that will take the contents of an email, and will suggest an appropriate short subject line for the email. That's the kind of feature that might be built into a commercial email tool.</span>
        </td>
    </tr>
</table>

In [ ]:
# Step 1: Create your prompts

system_prompt = "something here"
user_prompt = """
    Lots of text
    Can be pasted here
"""

# Step 2: Make the messages list

messages = [] # fill this in

# Step 3: Call OpenAI
# response =

# Step 4: print the result
# print(